In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.stattools import adfuller, pacf
from scipy.stats import boxcox, bartlett
from scipy.special import inv_boxcox
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ─────────────────────────────────────────
# STEP 1: Dataset
# Strong upward trend + growing seasonal swings
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
np.random.seed(0)
T = 36
trend    = np.array([100 + 5*t for t in range(T)])
seasonal = np.array([trend[t] * 0.15 * np.sin(2 * np.pi * t / 12) for t in range(T)])
noise    = np.random.normal(0, 8, T)
y        = np.round(trend + seasonal + noise).astype(int)

train_len = 30
y_train   = y[:train_len]
y_test    = y[train_len:]

print("=" * 60)
print("STEP 1: DATASET")
print("=" * 60)
print(f"T={T} months | Train={train_len} | Test={len(y_test)}")
print(f"y = {y}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(range(1, train_len+1), y_train, 'b-o', markersize=3, label='Train')
ax.plot(range(train_len+1, T+1), y_test, 'g-o', markersize=5, label='Test')
ax.axvline(x=train_len+0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title('Raw series — upward trend + growing seasonal swings')
ax.set_xlabel('Month'); ax.set_ylabel('Sales')
ax.legend(); plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/v4_step1_raw.png', dpi=120)
plt.close()
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 2: Check variance stability → Box-Cox?
# Split series into halves, compare variance
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: CHECK VARIANCE STABILITY → BOX-COX?")
print("=" * 60)
print("Split train into two halves, compare variance:")

half = len(y_train) // 2
first_half  = y_train[:half]
second_half = y_train[half:]

var1 = np.var(first_half)
var2 = np.var(second_half)
print(f"  Variance first half  (t=1  to t={half})  : {var1:.1f}")
print(f"  Variance second half (t={half+1} to t={train_len}) : {var2:.1f}")
print(f"  Ratio second/first = {var2/var1:.2f}")

# Bartlett's test: H0 = variances are equal
_, p_bartlett = bartlett(first_half.astype(float), second_half.astype(float))
print(f"\nBartlett's test p-value = {p_bartlett:.4f}")
print("H0: variances are equal")

if p_bartlett < 0.05 or var2/var1 > 2:
    apply_boxcox = True
    print("RESULT: Variance is UNSTABLE ❌ → apply Box-Cox")
else:
    apply_boxcox = False
    print("RESULT: Variance is STABLE ✅ → skip Box-Cox")
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 3: Apply Box-Cox if needed
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: BOX-COX TRANSFORMATION")
print("=" * 60)

if apply_boxcox:
    y_after_boxcox, lam = boxcox(y_train)
    print(f"Lambda λ = {lam:.4f}")
    print(f"std before = {np.std(y_train):.2f}  →  std after = {np.std(y_after_boxcox):.4f}")
    print("Variance stabilized ✅")
else:
    y_after_boxcox = y_train.astype(float)
    lam = None
    print("Skipped — variance was already stable")
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 4: ADF test on (possibly Box-Cox'd) series
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: ADF TEST → DIFFERENCING NEEDED?")
print("=" * 60)
print("H0: series is NON-stationary")
print("Rule: p > 0.05 → difference | p < 0.05 → skip differencing")

adf1 = adfuller(y_after_boxcox)
print(f"\nADF on {'Box-Cox series' if apply_boxcox else 'raw series'}:")
print(f"  ADF Statistic : {adf1[0]:.4f}")
print(f"  p-value       : {adf1[1]:.4f}")

if adf1[1] < 0.05:
    print("  RESULT: STATIONARY ✅ → skip differencing  d=0")
    y_after_diff = y_after_boxcox
    d = 0
else:
    print("  RESULT: NON-STATIONARY ❌ → apply 1st differencing")
    y_diff1 = np.diff(y_after_boxcox)

    adf2 = adfuller(y_diff1)
    print(f"\nADF after 1st differencing:")
    print(f"  ADF Statistic : {adf2[0]:.4f}")
    print(f"  p-value       : {adf2[1]:.4f}")

    if adf2[1] < 0.05:
        print("  RESULT: STATIONARY ✅  →  d=1 is enough")
        y_after_diff = y_diff1
        d = 1
    else:
        print("  RESULT: STILL NON-STATIONARY ❌ → apply 2nd differencing")
        y_diff2 = np.diff(y_diff1)

        adf3 = adfuller(y_diff2)
        print(f"\nADF after 2nd differencing:")
        print(f"  ADF Statistic : {adf3[0]:.4f}")
        print(f"  p-value       : {adf3[1]:.4f}")
        if adf3[1] < 0.05:
            print("  RESULT: STATIONARY ✅  →  d=2")
        else:
            print("  WARNING: still non-stationary — check data")
        y_after_diff = y_diff2
        d = 2

print(f"\nFinal d = {d}")
print(f"Series for model (length={len(y_after_diff)}): {np.round(y_after_diff, 4)}")
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 5: PACF → choose p
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: PACF → CHOOSE p")
print("=" * 60)

y_for_model = y_after_diff
max_lags    = min(10, len(y_for_model) // 2 - 1)
pacf_vals   = pacf(y_for_model, nlags=max_lags)
sig_bnd     = 2 / np.sqrt(len(y_for_model))

print(f"n={len(y_for_model)}  →  significance boundary = ±2/√{len(y_for_model)} = ±{sig_bnd:.3f}")
print(f"\n{'Lag':<6}{'PACF':>8}  {'Significant?'}")
print("-" * 32)

sig_lags = []
for lag in range(1, max_lags + 1):
    val    = pacf_vals[lag]
    is_sig = abs(val) > sig_bnd
    if is_sig:
        sig_lags.append(lag)
    print(f"{lag:<6}{val:>8.4f}  {'✅ YES' if is_sig else '❌ no'}")

# p = last consecutive significant lag starting from lag 1
p = 0
for lag in range(1, max_lags + 1):
    if lag in sig_lags:
        p = lag
    else:
        break
p = max(p, 1)

print(f"\nSignificant lags : {sig_lags}")
print(f"Chosen p         : {p}")

fig, ax = plt.subplots(figsize=(8, 3))
plot_pacf(y_for_model, lags=max_lags, ax=ax,
          color='steelblue', vlines_kwargs={'colors': 'steelblue'})
ax.set_title(f"PACF  |  boundary=±{sig_bnd:.2f}  |  chosen p={p}")
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/v4_step5_pacf.png', dpi=120)
plt.close()
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 6: Build X matrix
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print(f"STEP 6: BUILD X MATRIX  (p={p} lag columns)")
print("=" * 60)

def build_lag_matrix(series, p):
    X, y_out = [], []
    for i in range(p, len(series)):
        X.append(series[i-p:i][::-1])
        y_out.append(series[i])
    return np.array(X), np.array(y_out)

X_tr, y_tgt = build_lag_matrix(y_for_model, p)
print(f"X shape: {X_tr.shape}  →  {X_tr.shape[0]} rows, {p} lag features")
df_show = pd.DataFrame(np.round(X_tr, 4),
                       columns=[f'y(t-{i})' for i in range(1, p+1)])
df_show['→ target'] = np.round(y_tgt, 4)
print(df_show.to_string(index=False))
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 7: Fit Linear Regression
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: FIT LINEAR REGRESSION → COEFFICIENTS")
print("=" * 60)

model = LinearRegression()
model.fit(X_tr, y_tgt)

print(f"Intercept  α0 = {model.intercept_:.6f}")
for i, c in enumerate(model.coef_):
    print(f"Coeff      α{i+1} = {c:.6f}   (lag {i+1})")

eq = f"ŷ(t) = {model.intercept_:.4f}"
for i, c in enumerate(model.coef_):
    sign = "+" if c >= 0 else "-"
    eq  += f" {sign} {abs(c):.4f}×y(t-{i+1})"
print(f"\nAR({p}) equation:\n  {eq}")
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 8: Recursive prediction
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: PREDICT TEST SET (recursive)")
print("=" * 60)

last_vals         = list(y_for_model[-p:])
y_pred_model_space = []

for i in range(len(y_test)):
    x_in = np.array(last_vals[-p:][::-1]).reshape(1, -1)
    pred = model.predict(x_in)[0]
    y_pred_model_space.append(pred)
    last_vals.append(pred)
    print(f"  t={train_len+i+1}:  lags={np.round(last_vals[-p-1:-1][::-1], 4)}"
          f"  →  ŷ={pred:.6f}")
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 9: Reverse differencing (if d>0)
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: REVERSE DIFFERENCING")
print("=" * 60)

if d == 0:
    print("d=0 → no differencing was applied → skip this step")
    y_after_undiff = np.array(y_pred_model_space)

elif d == 1:
    anchor = y_after_boxcox[-1]
    print(f"d=1 → cumsum predictions starting from anchor={anchor:.4f}")
    y_after_undiff = []
    running = anchor
    for i, val in enumerate(y_pred_model_space):
        running += val
        y_after_undiff.append(running)
        print(f"  t={train_len+i+1}: {anchor:.4f} + cumsum → {running:.4f}")
    y_after_undiff = np.array(y_after_undiff)

elif d == 2:
    anchor_d1 = np.diff(y_after_boxcox)[-1]
    anchor_d0 = y_after_boxcox[-1]
    print(f"d=2 → two cumsum steps")
    print(f"  anchor d1 = {anchor_d1:.4f}  anchor d0 = {anchor_d0:.4f}")
    # undo 2nd diff
    y_undiff1 = []
    running = anchor_d1
    for val in y_pred_model_space:
        running += val
        y_undiff1.append(running)
    # undo 1st diff
    y_after_undiff = []
    running = anchor_d0
    for val in y_undiff1:
        running += val
        y_after_undiff.append(running)
    y_after_undiff = np.array(y_after_undiff)
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 10: Reverse Box-Cox (if applied)
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 10: REVERSE BOX-COX")
print("=" * 60)

if apply_boxcox:
    y_pred_final = inv_boxcox(y_after_undiff, lam)
    print(f"λ={lam:.4f} → inverse Box-Cox applied")
else:
    y_pred_final = y_after_undiff
    print("Box-Cox was not applied → skip this step")

print(f"\nPredicted sales : {np.round(y_pred_final, 1)}")
print(f"Actual sales    : {y_test}")
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
# STEP 11: Evaluate
# ─────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 11: EVALUATE")
print("=" * 60)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
mape = np.mean(np.abs(y_test - y_pred_final) / y_test) * 100
print(f"RMSE = {rmse:.2f}")
print(f"MAPE = {mape:.2f}%")

# Final plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, train_len+1), y_train,
        'b-o', markersize=3, label='Train (actual)')
ax.plot(range(train_len+1, T+1), y_test,
        'g-o', markersize=6, label='Test (actual)')
ax.plot(range(train_len+1, T+1), y_pred_final,
        'm--s', markersize=6, label=f'AR({p}) predictions')
ax.axvline(x=train_len+0.5, color='gray',
           linestyle='--', linewidth=1, label='Train/Test split')
ax.set_xlabel('Month'); ax.set_ylabel('Sales')
ax.set_title(f'AR({p}) | Box-Cox={apply_boxcox} | d={d} | '
             f'RMSE={rmse:.1f} | MAPE={mape:.1f}%')
ax.legend(); plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/v4_step11_final.png', dpi=120)
plt.close()
print("\nAll plots saved.")
print("=" * 60)
# ─────────────────────────────────────────
